### define gRPC service

// .proto: define service and message format

syntax = "proto3";

service RealTimeData {
    rpc GetRealTimeData(Empty) returns (stream DataPoint);
}

message DataPoint {
    int64 timestamp = 1;
    double value = 2;
}

message Empty {}

// 这个服务定义了一个 GetRealTimeData 方法，它返回一个 stream，表示实时数据的流。

// 使用 grpcio-tools 生成 Python 客户端和服务器代码。运行以下命令：
// .bash:
python -m grpc_tools.protoc -I. --python_out=. --grpc_python_out=. your_service.proto


In [ ]:

# 在服务器端，创建一个 RealTimeData 服务，模拟实时数据流的发送：
import time
import grpc
import your_service_pb2
import your_service_pb2_grpc
from concurrent import futures

class RealTimeDataServicer(your_service_pb2_grpc.RealTimeDataServicer):
    def GetRealTimeData(self, request, context):
        while True:
            data_point = your_service_pb2.DataPoint(timestamp=int(time.time()), value=42.0)
            yield data_point  # 不断发送实时数据
            time.sleep(1)

def serve():
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    your_service_pb2_grpc.add_RealTimeDataServicer_to_server(RealTimeDataServicer(), server)
    server.add_insecure_port('[::]:50051')
    server.start()
    print("Server started on port 50051")
    server.wait_for_termination()

if __name__ == '__main__':
    serve()


In [2]:
# 在客户端，连接到服务器并接收实时数据流：
import grpc
import your_service_pb2
import your_service_pb2_grpc

def run():
    channel = grpc.insecure_channel('localhost:50051')
    stub = your_service_pb2_grpc.RealTimeDataStub(channel)

    for data_point in stub.GetRealTimeData(your_service_pb2.Empty()):
        print(f"Timestamp: {data_point.timestamp}, Value: {data_point.value}")

if __name__ == '__main__':
    run()


ModuleNotFoundError: No module named 'your_service_pb2'

### 6. 启动服务器和客户端

// .bash:
python server.py
python client.py


### ref

https://www.helius.dev/docs/zh/grpc